Load the original CSV file.

In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"online_shoppers_intention_uncleaned.csv")
df.head()


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0.0,0.0,0.0,0.0,1.0,0.000000,0.20,0.20,0.0,0.0,Feb,1.0,1.0,1.0,1.0,Returning_Visitor,False,False
1,0.0,0.0,0.0,0.0,2.0,64.000000,0.00,0.10,0.0,0.0,Feb,2.0,2.0,1.0,2.0,Returning_Visitor,False,False
2,0.0,0.0,0.0,0.0,1.0,0.000000,0.20,0.20,0.0,0.0,Feb,4.0,1.0,9.0,3.0,Returning_Visitor,False,False
3,0.0,0.0,0.0,0.0,2.0,2.666667,0.05,0.14,0.0,0.0,Feb,3.0,2.0,2.0,4.0,Returning_Visitor,False,False
4,0.0,0.0,0.0,0.0,10.0,627.500000,0.02,0.05,0.0,0.0,Feb,3.0,3.0,1.0,4.0,Returning_Visitor,True,False


Remove duplicate rows

In [25]:
print("Shape",df.shape)
print("Duplicates:", df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)

Shape (12390, 18)
Duplicates: 91
Shape after removing duplicates: (12299, 18)


Handle missing values

In [ ]:
# Numerical columns
numeric_cols = df.select_dtypes(include="number").columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Category columns
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    if df[col].notna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values:", df.isnull().sum().sum())

Missing values: 0


C:\Users\battu\AppData\Local\Temp\ipykernel_20824\3589106284.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


Prepare the target column

In [27]:
# Convert Revenue into 0 and 1
df["Revenue"] = (
    df["Revenue"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0
    })
)

# Remove rows with invalid target values
df = df.dropna(subset=["Revenue"]).copy()
df["Revenue"] = df["Revenue"].astype(int)
print(df["Revenue"].value_counts())

Revenue
0    10389
1     1910
Name: count, dtype: int64


Save the cleaned dataset

In [28]:
df.to_csv("online_shoppers_cleaned.csv", index=False)

print("Preprocessed dataset saved!")
print("Final shape:", df.shape)

Preprocessed dataset saved!
Final shape: (12299, 18)


Feature selection

In [29]:
X = df.drop("Revenue", axis=1)
y = df["Revenue"]

# Remove columns with only one unique value
constant_cols = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

X = X.drop(columns=constant_cols)

# Encode categorical columns
X = pd.get_dummies(X, dtype=int)

print("Total available features:", X.shape[1])

Total available features: 48


Train-test split and feature selection

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 9839
Testing rows: 2460


Select the top 10 features

In [31]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

selector = SelectKBest(
    score_func=mutual_info_classif,
    k=min(10, X_train.shape[1])
)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]

print("Selected features:")
print(selected_features.tolist())

Selected features:
['Administrative', 'Administrative_Duration', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'TrafficType', 'Month_Mar']


Display feature scores

In [43]:
scores = pd.DataFrame({
    "Feature": X_train.columns,
    "Score": selector.scores_
})

scores = scores.sort_values(
    by="Score",
    ascending=False
)

print(scores.head(10))

                    Feature     Score
8                PageValues  0.166441
7                 ExitRates  0.041554
5   ProductRelated_Duration  0.030304
6               BounceRates  0.024109
4            ProductRelated  0.018421
0            Administrative  0.018108
1   Administrative_Duration  0.014877
13              TrafficType  0.014461
3    Informational_Duration  0.009778
27                Month_Mar  0.009153


Choose a machine learning model

In [33]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

print("Random Forest model created!")

Random Forest model created!


Model training

In [34]:
model.fit(X_train_selected, y_train)

print("Model training completed!")

Model training completed!


Prediction

In [35]:
y_pred = model.predict(X_test_selected)

print("Actual values:")
print(y_test.head(10).tolist())

print("Predicted values:")
print(y_pred[:10].tolist())

Actual values:
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Predicted values:
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Performance evaluation

In [39]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Accuracy: 0.8878048780487805
Precision: 0.7154471544715447
Recall: 0.4607329842931937
F1-score: 0.5605095541401274

Confusion Matrix:
[[2008   70]
 [ 206  176]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.97      0.94      2078
           1       0.72      0.46      0.56       382

    accuracy                           0.89      2460
   macro avg       0.81      0.71      0.75      2460
weighted avg       0.88      0.89      0.88      2460



Save your trained model

In [44]:
import joblib

joblib.dump(model, "purchase_prediction_model.pkl")
joblib.dump(selector, "feature_selector.pkl")
joblib.dump(list(X_train.columns), "feature_columns.pkl")

print("Model and feature selector saved!")

Model and feature selector saved!
